Az Ollama fusson, és legyen letöltve a `bge-m3` is (`ollama pull bge-m3`, kb. 1,2 GB). Futtasd sorban: **Shift + Enter**. Ha valami elakad: **Restart**, aztán **Run All**. A Copilot kódját olvasd el, mielőtt **Keep**-et nyomsz; ha rossz, **Undo**.

In [ ]:
import json
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymupdf

import kurzus
from kurzus import chat, embed, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Szövegből vektor: játék két dimenzióban

Két szám mondatonként: mennyire szól az árakról, és mennyire a munkaerőpiacról. **Kitalált számok**, a példa kedvéért.

In [ ]:
jatek = {                                       # kitalált példa: [árak, munkaerőpiac]
    "Csökkent az infláció.":                     np.array([0.9, 0.1]),
    "Lassult az áremelkedés.":                   np.array([0.8, 0.2]),
    "Nőtt a munkanélküliség.":                   np.array([0.1, 0.9]),
    "A bérek gyorsabban nőttek, mint az árak.":  np.array([0.6, 0.6]),
}

fig, ax = plt.subplots(figsize=(7, 4.5))
for mondat, v in jatek.items():
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops={"arrowstyle": "->", "color": "#252A51"})
    ax.text(v[0] + 0.02, v[1], mondat, fontsize=9)
ax.set_xlim(0, 1.6)
ax.set_ylim(0, 1.05)
ax.set_xlabel("mennyire szól az árakról")
ax.set_ylabel("mennyire szól a munkaerőpiacról")
plt.show()

A **koszinusz-hasonlóság** a két nyíl irányát hasonlítja össze: 1, ha ugyanarra mutatnak, 0, ha merőlegesek.

In [ ]:
def koszinusz(a, b):
    """Két vektor koszinusz-hasonlósága: 1 = ugyanarra mutatnak, 0 = merőlegesek."""
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

### Feladat 1.1 – olvasás

Mennyi a koszinusz-hasonlóság a `[3, 4]` és a `[6, 8]` vektor között? (Rajzold le fejben: merre mutatnak?)

In [ ]:
tipp_1_1 = None  # a None helyére írd a tipped

In [ ]:
ellenoriz("tipp_1_1", vart=float(koszinusz(np.array([3, 4]), np.array([6, 8]))))

A játékmondatok hasonlósága az elsőhöz:

In [ ]:
elso = jatek["Csökkent az infláció."]
for mondat, v in jatek.items():
    print(f"{koszinusz(elso, v):.3f}  {mondat}")

## 2. Igazi embedding: a `bge-m3`

Ugyanez 1024 számmal, a modell számolja (`kurzus.embed`):

In [ ]:
mondatok = [
    "Csökkent az infláció.",
    "Mérséklődött az áremelkedés üteme.",
    "Inflation has fallen.",
    "Nőtt a munkanélküliség.",
    "Gyengült a forint az euróval szemben.",
]
mondat_vektorok = np.array(embed(mondatok))
print("alak:", mondat_vektorok.shape)            # (mondatok száma, számok mondatonként)
print("az első mondat első 6 száma:", mondat_vektorok[0][:6].round(3))

Az összes pár hasonlósága egy mátrixszorzással (normalizálás után):

In [ ]:
normalt = mondat_vektorok / np.linalg.norm(mondat_vektorok, axis=1, keepdims=True)
hasonlosag = pd.DataFrame(normalt @ normalt.T, index=mondatok, columns=range(1, 6)).round(3)
hasonlosag

### Feladat 2.1 – Copilottal

> A `hasonlosag` táblában keresd meg, melyik **másik** mondat a leghasonlóbb az első mondathoz („Csökkent az infláció.”). A mondat szövegét tedd a `leghasonlobb` változóba, és írd ki.

Olvasd el: kihagyja a kód az első mondatot önmagával (ahol a hasonlóság 1)?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
_h = normalt @ normalt[0]
ellenoriz("leghasonlobb", vart=mondatok[int(np.argsort(-_h)[1])])

Ugyanez a közös szavak számával:

In [ ]:
def szavak(szoveg):
    """A szöveg szavainak halmaza, kisbetűsítve."""
    return set(re.findall(r"\w+", szoveg.lower()))


for i, mondat in enumerate(mondatok[1:], start=1):
    kozos = szavak(mondatok[0]) & szavak(mondat)
    print(f"közös szavak: {len(kozos)} {sorted(kozos)!s:12}  koszinusz: {normalt[0] @ normalt[i]:.3f}  {mondat}")

A kulcsszavas keresésnek az angol fordítás „idegen”, a jelentés szerinti keresésnek nem. A magyar ragozás (*infláció, inflációt, inflációs*) és a többnyelvűség miatt kell olyan embedding-modell, amely jól tud magyarul: a `bge-m3` ilyen.

## 3. A jelentések darabjainak indexelése

A 3. alkalom függvényei (új nincs benne, csak futtasd). Ha hiányzik a `data/cache/darabok.json`, ezekkel készül újra (kb. 10 mp).

In [ ]:
# A 3. alkalom függvényei: PDF → bekezdések → darabok. Csak futtasd.
FEJLEC_JELEK = ("INFLÁCIÓS JELENTÉS •", "MAGYAR NEMZETI BANK")
DARABOK_FAJL = Path("data/cache/darabok.json")


def tisztit(szoveg):
    """PDF-ből kinyert szöveg takarítása: elválasztás, pontsorok, sortörések, szóközök."""
    szoveg = re.sub(r"-\n(?=[a-záéíóöőúüű])", "", szoveg)  # sor végi elválasztás: ren-|delkezésére
    szoveg = re.sub(r"\.{4,}", " ", szoveg)                # tartalomjegyzék pontsora
    szoveg = szoveg.replace("\n", " ")
    return re.sub(r"\s+", " ", szoveg).strip()


def nyomtatott_oldalszam(oldal):
    """Az élőfejben álló nyomtatott oldalszám; ha az oldalon nincs élőfej (borító, belső borító), None."""
    elso_sorok = [s.strip() for s in oldal.get_text().splitlines()[:5]]
    if not any("INFLÁCIÓS JELENTÉS •" in s for s in elso_sorok):
        return None
    for sor in elso_sorok:
        if sor.isdigit():
            return int(sor)
    return None


def oldal_bekezdesei(oldal):
    """Az oldal bekezdései takarítva, élőfej, grafikonfeliratok és oldalszám nélkül."""
    bekezdesek = []
    for blokk in oldal.get_text("blocks"):
        szoveg = tisztit(blokk[4])
        if any(jel in szoveg for jel in FEJLEC_JELEK) or szoveg.isupper():
            continue
        if len(szoveg.split()) < 5 or not re.search(r"[a-zA-ZáéíóöőúüűÁÉÍÓÖŐÚÜŰ]{3}", szoveg):
            continue
        bekezdesek.append(szoveg)
    return bekezdesek


def jelentes_oldalai(pdf_utvonal, jelentes):
    """Egy jelentés oldalai: jelentés, PDF-oldalszám, nyomtatott oldalszám és a bekezdések."""
    return [{"jelentes": jelentes, "pdf_oldal": i, "oldal": nyomtatott_oldalszam(o), "bekezdesek": oldal_bekezdesei(o)}
            for i, o in enumerate(pymupdf.open(pdf_utvonal), start=1)]


def darabol_fix(szoveg, meret=1000, atfedes=0):
    """Fix méretű darabok (karakterben); az egymást követő darabok `atfedes` karakternyit átfednek."""
    lepes = meret - atfedes
    return [szoveg[i:i + meret] for i in range(0, max(len(szoveg) - atfedes, 1), lepes)]


ROVIDITESEK = {"pl", "kb", "stb", "ill", "ún", "vö", "szp", "Kr", "dr"}


def mondatokra(szoveg):
    """Mondatokra bontja a szöveget. Mondatvég: pont, kérdőjel vagy felkiáltójel, utána szóköz és
    nagybetű vagy számjegy; kivéve, ha a pont előtt szám, római szám, egyetlen betű vagy rövidítés áll."""
    mondatok = []
    for resz in re.split(r"(?<=[.!?]) (?=[A-ZÁÉÍÓÖŐÚÜŰ0-9])", szoveg):
        elozo_szo = mondatok[-1].split()[-1].rstrip(".") if mondatok else ""
        if elozo_szo.isdigit() or re.fullmatch(r"[IVX]+|\w", elozo_szo) or elozo_szo in ROVIDITESEK:
            mondatok[-1] += " " + resz
        else:
            mondatok.append(resz)
    return mondatok


def darabol_bekezdesenkent(bekezdesek, max_hossz=1000):
    """Egész bekezdéseket fűz össze, amíg a darab el nem érné a `max_hossz` karaktert. A túl hosszú
    bekezdést mondatokra bontja; ha egyetlen mondat is túl hosszú, azt szavakra."""
    reszek = []
    for bekezdes in bekezdesek:
        if len(bekezdes) <= max_hossz:
            reszek.append(bekezdes)
            continue
        for mondat in mondatokra(bekezdes):
            reszek += [mondat] if len(mondat) <= max_hossz else mondat.split()
    darabok, aktualis = [], ""
    for resz in reszek:
        if aktualis and len(aktualis) + 1 + len(resz) > max_hossz:
            darabok.append(aktualis)
            aktualis = resz
        else:
            aktualis = (aktualis + " " + resz).strip()
    if aktualis:
        darabok.append(aktualis)
    return darabok


def darabok_keszitese(oldalak, max_hossz=1000):
    """Egy jelentés darabjai; minden darab megőrzi a jelentést és mindkét oldalszámot."""
    darabok = []
    for o in oldalak:
        for j, szoveg in enumerate(darabol_bekezdesenkent(o["bekezdesek"], max_hossz)):
            darabok.append({"id": f'{o["jelentes"]}_p{o["pdf_oldal"]:03d}_{j}', "jelentes": o["jelentes"],
                            "pdf_oldal": o["pdf_oldal"], "oldal": o["oldal"], "szoveg": szoveg})
    return darabok


def darabok_betoltese(fajl=DARABOK_FAJL):
    """A forrasok.csv összes jelentésének darabjai a gyorsítótárból; ha nincs meg, újra elkészíti és elmenti (kb. 10 mp)."""
    if fajl.exists():
        return json.loads(fajl.read_text(encoding="utf-8"))
    print("Nincs meg a darabok.json: újra elkészítem a 3. alkalom függvényeivel.")
    darabok = []
    for s in pd.read_csv("data/mnb/forrasok.csv", sep=";").itertuples():
        darabok += darabok_keszitese(jelentes_oldalai(kurzus.letolt(s.url, "data/mnb/" + s.fajl), s.jelentes))
    fajl.parent.mkdir(parents=True, exist_ok=True)
    fajl.write_text(json.dumps(darabok, ensure_ascii=False), encoding="utf-8")
    return darabok

In [ ]:
osszes_darab = darabok_betoltese()

# Négy jelentés: két régi, egy 2025-ös és a legújabb. Az indok a következő cellákban.
JELENTESEK = ["2023-12", "2024-09", "2025-03", "2026-09"]
darabok = [d for d in osszes_darab if d["jelentes"] in JELENTESEK]
print(f"{len(osszes_darab)} darab összesen, ebből {len(darabok)} a kiválasztott {len(JELENTESEK)} jelentésben")

A beágyazás sebessége 64 darabon (pár másodperc):

In [ ]:
embed("bemelegítés")                              # betölti a bge-m3 modellt

proba = [d["szoveg"] for d in darabok[:64]]
kezdes = time.perf_counter()
embed(proba)                                      # egyetlen hívás, 64 szöveggel
proba_ido = time.perf_counter() - kezdes
sebesseg = len(proba) / proba_ido
print(f"64 darab: {proba_ido:.1f} mp, vagyis {sebesseg:.1f} darab/mp")

### Feladat 3.1 – Copilottal

> A `sebesseg` (darab/másodperc) alapján becsüld meg, hány másodpercig tartana az `osszes_darab` beágyazása (`becsult_ido_osszes`), és mennyi ideig a kiválasztott `darabok`-é (`becsult_ido_valasztott`). Írd ki mindkettőt percben is.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz("becsult_ido_osszes", vart=len(osszes_darab) / sebesseg, tures=1)
ellenoriz("becsult_ido_valasztott", vart=len(darabok) / sebesseg, tures=1)

Órán csak 4 jelentést indexelünk (idő). Otthon a `JELENTESEK`-be írhatod mind a 14-et. A beágyazás adagokban megy, az eredmény a `data/cache/`-be kerül (kb. 1 perc, ha még nincs meg; utána pillanatok).

In [ ]:
INDEX_FAJL = Path("data/cache/embeddingek.npz")


def beagyaz(szovegek, adag=64):
    """Szövegek embeddingje adagokban (egy hívás = `adag` szöveg); numpy-mátrixot ad vissza."""
    vektorok = []
    for i in range(0, len(szovegek), adag):
        vektorok += embed(szovegek[i:i + adag])
    return np.array(vektorok, dtype=np.float32)


def index_betoltese(darabok, fajl=INDEX_FAJL):
    """A darabok normalizált embedding-mátrixa: a gyorsítótárból, ha ugyanezekhez a darabokhoz
    készült, különben kiszámolja és elmenti. Az i. sor az i. darabé."""
    idk = [d["id"] for d in darabok]
    if fajl.exists():
        tarolt = np.load(fajl)
        if list(tarolt["idk"]) == idk:
            print(f"Gyorsítótárból: {len(idk)} darab (a kiszámolás {float(tarolt['ido']):.0f} mp volt).")
            return tarolt["vektorok"]
    kezdes = time.perf_counter()
    vektorok = beagyaz([d["szoveg"] for d in darabok])
    vektorok = vektorok / np.linalg.norm(vektorok, axis=1, keepdims=True)   # normalizálás
    ido = time.perf_counter() - kezdes
    fajl.parent.mkdir(parents=True, exist_ok=True)
    np.savez(fajl, vektorok=vektorok, idk=np.array(idk), ido=ido)
    print(f"Kiszámolva: {len(idk)} darab, {ido:.0f} mp ({len(idk) / ido:.1f} darab/mp).")
    return vektorok


index = index_betoltese(darabok)
print("Az index alakja:", index.shape)

A „vektortár”: egy mátrix és a darabok listája, ugyanabban a sorrendben.

## 4. Keresés

A kérdés vektora, a hasonlóság minden darabhoz egy mátrixszorzással, és a `k` legjobb, a leghasonlóbbal kezdve. Előbb egy kiíró függvény:

In [ ]:
def kiir(talalatok):
    """A találatok kiírása: hasonlóság, forrás, a szöveg eleje."""
    for hasonlosag, d in talalatok:
        hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
        print(f'{hasonlosag:.3f}  {d["jelentes"]}, {hol}:  {d["szoveg"][:140]}…')

### Feladat 4.1 – hibakeresés

A Copilot első változata. Lefut, de valami nincs rendben. Futtasd: illenek a találatok a kérdéshez? Mekkorák a hasonlóságok?

In [ ]:
def keres_v1(kerdes, darabok, index, k=5):
    q = np.array(embed(kerdes)[0])
    q = q / np.linalg.norm(q)
    hasonlosagok = index @ q
    legjobbak = np.argsort(hasonlosagok)[:k]
    return [(float(hasonlosagok[i]), darabok[i]) for i in legjobbak]


kiir(keres_v1("Mekkora az MNB inflációs célja?", darabok, index))

Javítsd ki: írd meg a `keres(kerdes, darabok, index, k=5)` függvényt, amely a `k` **leghasonlóbb** darabot adja vissza, a leghasonlóbbal kezdve. (Az `np.argsort` növekvő sorrendet ad.)

In [ ]:
# Ide kerül a javítás

In [ ]:
if "keres" in globals():
    _t = keres("Mekkora az MNB inflációs célja?", darabok, index)
    _q = np.array(embed("Mekkora az MNB inflációs célja?")[0])
    _osszes = index @ (_q / np.linalg.norm(_q))            # a hasonlóság minden darabhoz
    _h = [h for h, _ in _t]
    keres_jo = len(_t) == 5 and abs(_h[0] - _osszes.max()) < 1e-4 and _h == sorted(_h, reverse=True)
else:
    keres_jo = None
ellenoriz_feltetel("keres_jo", lambda x: x, "5 találat, az első a leghasonlóbb darab, és a sorrend csökkenő")

Az én változatom (ha a 4.1 még nincs kész, ezzel megy tovább a notebook):

In [ ]:
def keres(kerdes, darabok, index, k=5):
    """A kérdéshez leghasonlóbb k darab, (hasonlóság, darab) párokként, csökkenő sorrendben."""
    q = np.array(embed(kerdes)[0])
    q = q / np.linalg.norm(q)
    hasonlosagok = index @ q                       # koszinusz-hasonlóság az összes darabbal egyszerre
    legjobbak = np.argsort(-hasonlosagok)[:k]      # a k legnagyobb hasonlóság helye
    return [(float(hasonlosagok[i]), darabok[i]) for i in legjobbak]

A várt oldalakat egy script a nyers PDF-ből keresi ki:

In [ ]:
forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")


def oldalak_ahol(minta, jelentesek=JELENTESEK):
    """(jelentés, PDF-oldal) párok, ahol a nyers PDF-szövegben megvan a minta (reguláris kifejezés)."""
    talalt = set()
    for s in forrasok[forrasok["jelentes"].isin(jelentesek)].itertuples():
        for i, oldal in enumerate(pymupdf.open("data/mnb/" + s.fajl), start=1):
            if re.search(minta, tisztit(oldal.get_text())):
                talalt.add((s.jelentes, i))
    return talalt


def van_jo_talalat(talalatok, vart_oldalak):
    """Igaz, ha a találatok között van olyan darab, amely a várt oldalak egyikéről jön."""
    return any((d["jelentes"], d["pdf_oldal"]) in vart_oldalak for _, d in talalatok)

### Első kérdés: az inflációs cél

In [ ]:
KERDES_CEL = "Mekkora az MNB inflációs célja?"
talalatok_cel = keres(KERDES_CEL, darabok, index)
kiir(talalatok_cel)

### Feladat 4.2 – olvasás

Melyik jelentésből jön az **első** találat? Írd be idézőjelek közé (például `"2024-09"`).

In [ ]:
tipp_4_2 = None  # a None helyére írd a jelentést, idézőjelek közé

In [ ]:
ellenoriz("tipp_4_2", vart=talalatok_cel[0][1]["jelentes"])

vart_cel = oldalak_ahol(r"középtávú (inflációs )?cél")
print("A nyers PDF-ekben ezeken az oldalakon áll a középtávú cél:", sorted(vart_cel))
ellenoriz_feltetel("talalatok_cel", lambda t: van_jo_talalat(t, vart_cel),
                   "az első 5 találat között van olyan oldal, amelyen a középtávú cél áll")

Mit mondanak a találatok, és melyik időpontra?

### Második kérdés: az infláció 2024-ben

In [ ]:
KERDES_2024 = "Mekkora volt az infláció 2024-ben?"
talalatok_2024 = keres(KERDES_2024, darabok, index)
kiir(talalatok_2024)

In [ ]:
# A tényadat helye a nyers PDF-ekben (az előző alkalom 5.1-es feladatából ismerős mondat).
vart_2024 = oldalak_ahol(r"2024-ben az infláció 3,7 százalékon alakult")
print("A várt oldal:", vart_2024)
helyezes = [i + 1 for i, (_, d) in enumerate(talalatok_2024) if (d["jelentes"], d["pdf_oldal"]) in vart_2024]
print("Hányadik találat a jó oldal:", helyezes)
ellenoriz_feltetel("talalatok_2024", lambda t: van_jo_talalat(t, vart_2024),
                   "a 2024-es tényadat oldala az első 5 találat között van")

A jó oldal az első. Olvasd el a többi négyet is: mit mondanak 2024-ről, és melyik jelentésből valók?

### Feladat 4.3 – Copilottal

> Számold meg, a `talalatok_2024` közül hány darab jön 2025 előtt megjelent jelentésből (a darab `jelentes` mezője kisebb, mint `"2025"`). Az eredmény a `regi_db` változóba kerüljön, és írd ki ezeknek a daraboknak az elejét is.

Olvasd el: szövegeket hasonlít össze a kód (`<`)? Működik ez a `"2023-12"` alakú értékekkel?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
ellenoriz("regi_db", vart=sum(d["jelentes"] < "2025" for _, d in talalatok_2024))

Olvasd el a 3–5. találatot: előrejelzés vagy tényadat? Melyik jelentésből? Mit válaszolna a modell, ha ezekből dolgozna, és mi lenne a hivatkozása? Mekkora a különbség az első és az ötödik hasonlósága között? Erre a következő két alkalommal visszatérünk (dátum a darab mellett, szűrés, script, amely a számot a nyers adatból ellenőrzi).

Ugyanez a GDP-vel. Hányadik a tényadat, és mi van előtte?

In [ ]:
KERDES_GDP = "Mennyivel bővült a magyar gazdaság 2024-ben?"
talalatok_gdp = keres(KERDES_GDP, darabok, index)
kiir(talalatok_gdp)

vart_gdp = oldalak_ahol(r"2024-ben a hazai GDP (\d,\d) százalékkal emelkedett")    # a tényadat helye a nyers PDF-ben
print("\nA tényadat oldala:", vart_gdp, "→ hányadik találat:",
      [i + 1 for i, (_, d) in enumerate(talalatok_gdp) if (d["jelentes"], d["pdf_oldal"]) in vart_gdp])

### Kulcsszavas vagy jelentés szerinti keresés?

A kulcsszavas keresés a kérdés és a darab közös szavait számolja:

In [ ]:
def kulcsszavas_kereses(kerdes, darabok, k=5):
    """A k darab, amelyben a kérdés szavai közül a legtöbb előfordul, (közös szavak száma, darab) párokként."""
    kerdes_szavai = szavak(kerdes)
    pontszamok = [len(kerdes_szavai & szavak(d["szoveg"])) for d in darabok]
    legjobbak = sorted(range(len(darabok)), key=lambda i: -pontszamok[i])[:k]
    return [(pontszamok[i], darabok[i]) for i in legjobbak]

Ugyanaz a kérdés angolul (a jelentések magyarok):

In [ ]:
KERDES_ANGOL = "What is the central bank's inflation target?"

print("KULCSSZAVAS:")
for pont, d in kulcsszavas_kereses(KERDES_ANGOL, darabok)[:3]:
    print(f"  {pont} közös szó {sorted(szavak(KERDES_ANGOL) & szavak(d['szoveg']))}: {d['jelentes']}, {d['szoveg'][:90]}…")
print("JELENTÉS SZERINTI:")
kiir(keres(KERDES_ANGOL, darabok, index, k=3))

### Feladat 4.4 – olvasás

Melyik az a közös szó a fenti kulcsszavas találatokban, amely magyarul is, angolul is szó, de mást jelent? Írd be idézőjelek közé.

In [ ]:
tipp_4_4 = None  # a None helyére írd a szót, idézőjelek közé

In [ ]:
# A kulcsszavas találatok közös szavai közül az, amelyik magyar szövegben is gyakori szó.
_kozos = set().union(*(szavak(KERDES_ANGOL) & szavak(d["szoveg"]) for _, d in kulcsszavas_kereses(KERDES_ANGOL, darabok)))
ellenoriz_feltetel("tipp_4_4", lambda t: t in _kozos and t in {"is", "a", "az"},
                   "ez a szó a magyar szövegben névelő vagy kötőszó, az angolban más")

A három kérdés egy táblázatban: van-e jó oldal az első 5 között?

In [ ]:
tesztek = [
    (KERDES_CEL, vart_cel),
    (KERDES_2024, vart_2024),
    (KERDES_ANGOL, vart_cel),
]
sorok = []
for kerdes, vart in tesztek:
    sorok.append({"kérdés": kerdes,
                  "kulcsszavas": van_jo_talalat(kulcsszavas_kereses(kerdes, darabok), vart),
                  "jelentés szerinti": van_jo_talalat(keres(kerdes, darabok, index), vart)})
pd.DataFrame(sorok).set_index("kérdés")

Miért bukik el a kulcsszavas keresés a két magyar kérdésnél is? Nézd meg a `kulcsszavas_kereses` találatait az inflációs célra: melyik bekezdést találja, és benne van-e a cél száma?

## 5. Mennyi fér a kontextusba?

A kérés (darabok + kérdés) maradjon **3000 token alatt**, hogy a 4096-os ablakban maradjon hely a gondolkodásnak és a válasznak. Mérjük meg az öt találatot a kérdéssel (egy hívás, pár mp):

In [ ]:
def tokenszam(szoveg):
    """Hány token a szöveg a helyi modell szerint. Egyetlen tokennyi választ kérünk, hogy gyors legyen."""
    chat([{"role": "user", "content": szoveg}], gondolkodas="none", max_tokens=1)
    return utolso_hasznalat["prompt_tokens"]


kontextus = "\n\n".join(d["szoveg"] for _, d in talalatok_2024)
keres_szovege = kontextus + "\n\nKérdés: " + KERDES_2024
keres_tokenek = tokenszam(keres_szovege)
arany = len(keres_szovege) / keres_tokenek
print(f"5 darab + kérdés: {len(keres_szovege)} karakter, {keres_tokenek} token ({arany:.1f} karakter/token)")

### Feladat 5.1 – Copilottal

Legrosszabb esettel számolj: minden darab olyan hosszú, mint a leghosszabb.

> Az `osszes_darab` leghosszabb darabjának hossza (karakter) és az `arany` (karakter/token) alapján számold ki, legfeljebb hány darab (`max_k`, egész szám, lefelé kerekítve) fér el 3000 tokenben, ha a kérdésre és az utasításokra 200 tokent hagyunk.

Olvasd el: lefelé kerekít a kód (`int` vagy `//`), és levonja a 200 tokent?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
_leghosszabb = max(len(d["szoveg"]) for d in osszes_darab)
ellenoriz("max_k", vart=int((3000 - 200) / (_leghosszabb / arany)))

Néhány `k` értékre:

In [ ]:
atlagos_hossz = sum(len(d["szoveg"]) for d in osszes_darab) / len(osszes_darab)
leghosszabb_hossz = max(len(d["szoveg"]) for d in osszes_darab)
pd.DataFrame([{"k": k,
               "token, átlagos darabokkal": round(k * atlagos_hossz / arany),
               "token, leghosszabb darabokkal": round(k * leghosszabb_hossz / arany),
               "belefér 3000 alá (legrosszabb eset)": k * leghosszabb_hossz / arany + 200 <= 3000}
              for k in [1, 3, 5, 8, 10, 20]]).set_index("k")

A következő alkalommal **k = 5** darabbal dolgozunk: a legrosszabb esetben is 3000 token alatt marad, és ha a jó darab csak a második-harmadik, még bekerül.